# Integer Square Root

The function `root(n)` takes a natural number $n$ and returns the largest natural number $r$ such that
$r^2 \leq n$, i.e. we have
$$ \texttt{isqrt}(n) := \max\bigl(\{ r \in \mathbb{N} \mid r^2 \leq n \}\bigr). $$
Our goal is to compute `root(n)` recursively via a *divide-and-conquer* algorithm as follows:
1. $\texttt{root}(0) = 0$.
2. $\bigl(2 \cdot \texttt{root}(n \,\texttt{//}\, 4) + 1\bigr)^2 \leq n \rightarrow \texttt{root}(n) = 2 \cdot \texttt{root}(n \,\texttt{//}\, 4) + 1$.
3. $\bigl(2 \cdot \texttt{root}(n \,\texttt{//}\, 4) + 1\bigr)^2 > n \rightarrow \texttt{root}(n) = 2 \cdot \texttt{root}(n \,\texttt{//}\, 4)$.

We use the class `BigInteger` from the package `java.math`, which supports integers of arbitrary size.  The constant `FOUR` is defined for convenience.

In [ ]:
BigInteger FOUR = BigInteger.valueOf(4);

BigInteger root(BigInteger n) {
    if (n.signum() == 0) {
        return BigInteger.ZERO;
    }
    BigInteger r  = root(n.divide(FOUR));
    BigInteger r1 = BigInteger.TWO.multiply(r).add(BigInteger.ONE);   // 2 * r + 1
    if (r1.pow(2).compareTo(n) <= 0) {
        return r1;
    } else {
        return BigInteger.TWO.multiply(r);
    }
}

In [ ]:
for (int n = 0; n < 18; n++) {
    System.out.println("root(" + n + ") = " + root(BigInteger.valueOf(n)));
}

In order to test our implementation more thoroughly we use random numbers.

Given a `BigInteger` `n`, the method `n.toString(2)` converts `n` into a binary string.

In [ ]:
BigInteger.valueOf(12345678).toString(2)

Conversely, the constructor `new BigInteger(s, 2)` converts the binary string `s` into a `BigInteger`.

In [ ]:
new BigInteger("101111000110000101001110", 2)

The function `randomBigIntBelow(max)` returns a random `BigInteger` that is less than `max`.  The constructor `new BigInteger(numBits, random)` creates a random number that has at most `numBits` binary digits.

In [ ]:
Random random = new Random();

BigInteger randomBigIntBelow(BigInteger max) {
    int maxBits = max.bitLength();
    BigInteger result;
    do {
        result = new BigInteger(maxBits, random);
    } while (result.compareTo(max) >= 0);
    return result;
}

The function `runTests(noTests)` generates `noTests` random integers `n` and tests whether
`root(n)` is the *integer square root* of `n` in each case.

In [ ]:
void runTests(int noTests) {
    for (int i = 0; i < noTests; i++) {
        BigInteger n  = randomBigIntBelow(BigInteger.TWO.pow(64));
        BigInteger r  = root(n);
        BigInteger r1 = r.add(BigInteger.ONE);
        if (!(r.multiply(r).compareTo(n) <= 0 && r1.multiply(r1).compareTo(n) > 0)) {
            throw new AssertionError("Error: " + r + " != isqrt(" + n + ")");
        }
    }
}

In [ ]:
long start = System.currentTimeMillis();
runTests(1_000_000);
System.out.println("runTests: " + (System.currentTimeMillis() - start) + " ms");

The function `sqrt2(k)` returns $\sqrt{2}$ up to `k` decimal places.

In [ ]:
String sqrt2(int k) {
    BigInteger n = BigInteger.TWO.multiply(BigInteger.TEN.pow(2 * k));
    BigInteger r = root(n);
    String     s = r.toString();
    return s.charAt(0) + "." + s.substring(1);
}

Let us compute the first $10\,000$ digits of $\sqrt{2\,}$.  For this computation, the recursion depth of `root` is about $\log_4\bigl(2 \cdot 10^{20\,000}\bigr) \approx 33\,000$.  The stack of the main thread of the Java virtual machine is too small for this recursion depth: The call `sqrt2(10_000)` would throw a `StackOverflowError`.  Therefore, we perform the computation in a new thread that has a stack of $2^{28}$ bytes, i.e. 256 MB.  The method `join` waits until this thread has finished.

In [ ]:
long start = System.currentTimeMillis();
String[] digits = new String[1];
Thread worker = new Thread(null, () -> digits[0] = sqrt2(10_000), "sqrt2", 1L << 28);
worker.start();
worker.join();
System.out.println(digits[0]);
System.out.println("sqrt2: " + (System.currentTimeMillis() - start) + " ms");

You can compare this with the results shown on https://catonmat.net/tools/generate-sqrt2-digits.